<a href="https://colab.research.google.com/github/CHARANSAI2003/20CS713-PP-1/blob/main/Final_NL_%3ESQL_%3ENL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cell 1: Install and Import Required Packages

In [ ]:
# Cell 1: Install and Import Required Packages
print("🚀 Setting up Natural Language ↔ SQL ↔ Natural Language System")
print("="*65)

# Install required packages (quiet mode to reduce output)
!pip install -q transformers torch sentencepiece pandas sqlalchemy accelerate

# Import essential libraries
import torch
import pandas as pd
import sqlite3
import warnings
import json
from datetime import datetime
from transformers import T5Tokenizer, T5ForConditionalGeneration, AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import files

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Check system resources
print("💻 System Information:")
print(f"✅ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   🔥 GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"   💾 GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   💻 Running on CPU")

print("\n✅ All packages installed successfully!")
print("📊 Total system will use <1GB disk space")


🚀 Setting up Natural Language ↔ SQL ↔ Natural Language System
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 63.0 MB/s eta 0:00:00
💻 System Information:
✅ GPU Available: False
   💻 Running on 

## Cell 2: Database Setup - Upload cleaned_studentdb.sql or Create Sample

In [ ]:
# Cell 2: Database Setup - Upload cleaned_studentdb.sql or Create Sample
print("📁 Database Setup")
print("="*35)

DATABASE_FILENAME = "studentdb.db"

# Upload your cleaned_studentdb.sql file
print("📤 Please upload your cleaned_studentdb.sql file:")
print("(If you don't upload, a sample database will be created)")
uploaded = files.upload()

if uploaded:
    # Use uploaded SQL file
    sql_filename = list(uploaded.keys())[0]
    print(f"✅ Uploaded: {sql_filename}")

    # Create SQLite database from SQL dump
    print("🔄 Creating SQLite database from your file...")
    conn = sqlite3.connect(DATABASE_FILENAME)

    try:
        with open(sql_filename, 'r', encoding='utf-8') as f:
            sql_content = f.read()

        # Execute SQL commands to create tables and insert data
        conn.executescript(sql_content)
        conn.commit()
        print("✅ Database created successfully from your SQL file!")

    except Exception as e:
        print(f"❌ Error creating database: {e}")
        print("🔄 Creating sample database instead...")
        conn = None

else:
    print("⚠️ No file uploaded. Creating sample database...")
    conn = None


📁 Database Setup
📤 Please upload your cleaned_studentdb.sql file:
(If you don't upload, a sample database will be created)


Saving cleaned_studentdb.sql to cleaned_studentdb.sql
✅ Uploaded: cleaned_studentdb.sql
🔄 Creating SQLite database from your file...
✅ Database created successfully from your SQL file!


## Show database overview

In [ ]:
# Show database overview
print(f"\n📊 Database Overview:")
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print(f"📋 Found {len(tables)} tables:")

for table_name in tables['name']:
    try:
        df = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 3", conn)
        print(f"\n📊 Table: {table_name} ({len(pd.read_sql(f'SELECT * FROM {table_name}', conn))} rows)")
        print(df.to_string(index=False))
    except Exception as e:
        print(f"⚠️ Could not preview {table_name}: {e}")



📊 Database Overview:
📋 Found 6 tables:

📊 Table: courses (4 rows)
 course_id     course_name  credit_hours  dept_id  instructor_id
         1 Data Structures             4        1              1
         2  Circuit Theory             3        2              2
         3  Thermodynamics             3        3              3

📊 Table: departments (4 rows)
 dept_id              dept_name   location
       1       Computer Science Building A
       2 Electrical Engineering Building B
       3 Mechanical Engineering Building C

📊 Table: enrollments (5 rows)
 student_id  course_id enrollment_date
          1          1      2024-08-15
          1          4      2024-08-19
          2          2      2024-08-16

📊 Table: grades (5 rows)
 grade_id  student_id  course_id grade  exam_date
        1           1          1     A 2024-12-05
        2           2          2     B 2024-12-10
        3           3          1     A 2024-12-12

📊 Table: instructors (4 rows)
 instructor_id         nam

## Cell 3: Extract and Analyze Database Schema

In [ ]:
# Cell 3: Extract and Analyze Database Schema
print("🗂️ Database Schema Analysis")
print("="*40)

def extract_database_schema(connection):
    """Extract comprehensive database schema information"""
    schema_info = {}
    cursor = connection.cursor()

    try:
        # Get all tables
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        tables = [table[0] for table in cursor.fetchall()]

        for table in tables:
            schema_info[table] = {
                'columns': [],
                'sample_values': {},
                'row_count': 0
            }

            # Get column information
            cursor.execute(f"PRAGMA table_info({table});")
            columns = cursor.fetchall()
            # Format: (cid, name, type, notnull, dflt_value, pk)
            schema_info[table]['columns'] = [(col[1], col[2], col[3], col[5]) for col in columns]

            # Get row count
            cursor.execute(f"SELECT COUNT(*) FROM {table};")
            schema_info[table]['row_count'] = cursor.fetchone()[0]

            # Get sample values for context
            for col_name, col_type, not_null, is_pk in schema_info[table]['columns']:
                try:
                    cursor.execute(f"SELECT DISTINCT {col_name} FROM {table} WHERE {col_name} IS NOT NULL LIMIT 3;")
                    samples = [str(row[0]) for row in cursor.fetchall()]
                    schema_info[table]['sample_values'][col_name] = samples
                except:
                    schema_info[table]['sample_values'][col_name] = []

        return schema_info

    except Exception as e:
        print(f"❌ Error extracting schema: {e}")
        return {}

def print_schema_summary(schema_info):
    """Print formatted schema summary"""
    print("📋 Database Schema Summary:")
    print("-" * 50)

    for table, info in schema_info.items():
        print(f"\n📊 Table: {table} ({info['row_count']} rows)")

        for col_name, col_type, not_null, is_pk in info['columns']:
            pk_indicator = " 🔑" if is_pk else ""
            null_indicator = " (NOT NULL)" if not_null else ""
            samples = info['sample_values'][col_name][:2]
            sample_str = f" | Examples: {', '.join(samples)}" if samples else ""
            print(f"   • {col_name} ({col_type}){pk_indicator}{null_indicator}{sample_str}")

# Extract schema from database
database_schema = extract_database_schema(conn)
print_schema_summary(database_schema)

# Create schema text for model prompts
def create_schema_text(schema_info):
    """Create schema text for model prompts"""
    schema_lines = []
    for table, info in schema_info.items():
        columns = []
        for col_name, col_type, not_null, is_pk in info['columns']:
            col_def = f"{col_name} {col_type}"
            if is_pk:
                col_def += " PRIMARY KEY"
            columns.append(col_def)

        create_statement = f"CREATE TABLE {table} ({', '.join(columns)})"
        schema_lines.append(create_statement)

    return "tables:\n" + "\n".join(schema_lines)

schema_text = create_schema_text(database_schema)
print(f"\n✅ Schema extracted successfully!")
print(f"📊 Tables: {len(database_schema)}, Total columns: {sum(len(info['columns']) for info in database_schema.values())}")


🗂️ Database Schema Analysis
📋 Database Schema Summary:
--------------------------------------------------

📊 Table: courses (4 rows)
   • course_id (INTEGER) 🔑 | Examples: 1, 2
   • course_name (TEXT) (NOT NULL) | Examples: Data Structures, Circuit Theory
   • credit_hours (INTEGER) | Examples: 4, 3
   • dept_id (INTEGER) | Examples: 1, 2
   • instructor_id (INTEGER) | Examples: 1, 2

📊 Table: departments (4 rows)
   • dept_id (INTEGER) 🔑 | Examples: 1, 2
   • dept_name (TEXT) (NOT NULL) | Examples: Computer Science, Electrical Engineering
   • location (TEXT) | Examples: Building A, Building B

📊 Table: enrollments (5 rows)
   • student_id (INTEGER) 🔑 | Examples: 1, 2
   • course_id (INTEGER) 🔑 | Examples: 1, 4
   • enrollment_date (TEXT) | Examples: 2024-08-15, 2024-08-19

📊 Table: grades (5 rows)
   • grade_id (INTEGER) 🔑 | Examples: 1, 2
   • student_id (INTEGER) | Examples: 1, 2
   • course_id (INTEGER) | Examples: 1, 2
   • grade (TEXT) | Examples: A, B
   • exam_date (TEXT) | Ex

## Cell 4: Load Optimized Text-to-SQL Model

In [ ]:
# Cell 4: Load Optimized Text-to-SQL Model
print("🤖 Loading Text-to-SQL Model")
print("="*40)

# Using cssupport/t5-small-awesome-text-to-sql - optimized for multi-table schemas
MODEL_NAME = "cssupport/t5-small-awesome-text-to-sql"

print(f"📥 Loading model: {MODEL_NAME}")
print("💡 This model is specifically optimized for multi-table schemas with JOINs")

try:
    # Load tokenizer and model
    print("📝 Loading tokenizer...")
    sql_tokenizer = T5Tokenizer.from_pretrained('t5-small')  # Use base T5 tokenizer

    print("🧠 Loading model...")
    sql_model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME)

    # Set up device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    sql_model = sql_model.to(device)
    sql_model.eval()  # Set to evaluation mode

    print(f"✅ Text-to-SQL model loaded successfully!")
    print(f"🔧 Device: {device}")
    print(f"📊 Model parameters: {sum(p.numel() for p in sql_model.parameters()):,}")
    print(f"💾 Model size: ~60MB (perfect for Colab free tier)")

    # Test tokenizer
    test_input = "SELECT * FROM students"
    test_tokens = sql_tokenizer(test_input, return_tensors="pt")
    print(f"🧪 Tokenizer test: {test_tokens['input_ids'].shape[1]} tokens")

except Exception as e:
    print(f"❌ Error loading text-to-SQL model: {e}")
    print("💡 Trying alternative approach...")

    try:
        # Fallback to direct AutoModel approach
        from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
        sql_tokenizer = AutoTokenizer.from_pretrained("t5-small")
        sql_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        sql_model = sql_model.to(device)
        print("✅ Loaded with fallback approach!")

    except Exception as e2:
        print(f"❌ Fallback failed: {e2}")
        print("⚠️ Will try with basic T5 model...")
        sql_tokenizer = T5Tokenizer.from_pretrained('t5-small')
        sql_model = T5ForConditionalGeneration.from_pretrained('t5-small')
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        sql_model = sql_model.to(device)
        print("✅ Using basic T5 model as fallback")

# Memory check
if torch.cuda.is_available():
    memory_used = torch.cuda.memory_allocated() / 1e9
    print(f"🔥 GPU memory used: {memory_used:.2f} GB")


🤖 Loading Text-to-SQL Model
📥 Loading model: cssupport/t5-small-awesome-text-to-sql
💡 This model is specifically optimized for multi-table schemas with JOINs
📝 Loading tokenizer...


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


🧠 Loading model...
✅ Text-to-SQL model loaded successfully!
🔧 Device: cpu
📊 Model parameters: 60,506,624
💾 Model size: ~60MB (perfect for Colab free tier)
🧪 Tokenizer test: 7 tokens


## Cell 5: Load SQL Results to Natural Language Model

In [ ]:
# Cell 5: Load SQL Results to Natural Language Model
print("🗣️ Loading Natural Language Generation Model")
print("="*50)

# Using FLAN-T5-small for converting SQL results back to natural language
NL_MODEL_NAME = "google/flan-t5-small"

try:
    print(f"📥 Loading model: {NL_MODEL_NAME}")
    nl_tokenizer = T5Tokenizer.from_pretrained(NL_MODEL_NAME)
    nl_model = T5ForConditionalGeneration.from_pretrained(NL_MODEL_NAME)

    # Move to device
    nl_model = nl_model.to(device)
    nl_model.eval()

    print("✅ Natural language generation model loaded successfully!")

    # Model information
    nl_params = sum(p.numel() for p in nl_model.parameters())
    print(f"📊 NL Model parameters: {nl_params:,}")
    print(f"💾 NL Model size: ~130MB")

    # Combined memory usage
    total_params = sum(p.numel() for p in sql_model.parameters()) + nl_params
    print(f"🎯 Combined models: ~{total_params / 1e6:.0f}M parameters (~200MB total)")

except Exception as e:
    print(f"❌ Error loading NL model: {e}")
    print("💡 Continuing with SQL-only functionality...")
    nl_model = None
    nl_tokenizer = None

# Final memory check
if torch.cuda.is_available():
    memory_used = torch.cuda.memory_allocated() / 1e9
    print(f"🔥 Total GPU memory used: {memory_used:.2f} GB")

print("\n🎯 Model Loading Summary:")
print(f"   📊 Text-to-SQL Model: {'✅ Loaded' if sql_model else '❌ Failed'}")
print(f"   🗣️ NL Generation Model: {'✅ Loaded' if nl_model else '❌ Failed'}")


🗣️ Loading Natural Language Generation Model
📥 Loading model: google/flan-t5-small
✅ Natural language generation model loaded successfully!
📊 NL Model parameters: 76,961,152
💾 NL Model size: ~130MB
🎯 Combined models: ~137M parameters (~200MB total)

🎯 Model Loading Summary:
   📊 Text-to-SQL Model: ✅ Loaded
   🗣️ NL Generation Model: ✅ Loaded


## Cell 6: Natural Language to SQL Conversion Function

In [ ]:
# Cell 6: Enhanced Natural Language to SQL Conversion Function
print("🔄 Enhanced Natural Language to SQL Converter")
print("="*50)

def get_reliable_sql(question, schema_info):
    q = question.lower()

    # 1. Students enrolled in a particular course (covers 'Data Science', can add more)
    from re import search
    if 'students' in q and ('enrolled in' in q or 'taking' in q):
        match = search(r'enrolled in ([\w\s]+)', q)
        course = match.group(1).strip() if match else "Data Science"
        return f"""
        SELECT s.name
        FROM students s
        JOIN enrollments e ON s.student_id = e.student_id
        JOIN courses c ON e.course_id = c.course_id
        WHERE lower(c.course_name) = '{course.lower()}'
        """

    # 2. Student names and emails in a department
    if 'students' in q and 'department' in q and ('name' in q or 'email' in q):
        match = search(r'in the ([\w\s]+) department', q)
        dept = match.group(1).strip() if match else "Computer Science"
        return f"""
        SELECT s.name, s.email
        FROM students s
        JOIN departments d ON s.dept_id = d.dept_id
        WHERE lower(d.dept_name) = '{dept.lower()}'
        """

    # 3. List all courses for a department
    if 'courses' in q and ('offered by' in q or 'by the' in q or 'department' in q):
        match = search(r'by the ([\w\s]+) department', q)
        dept = match.group(1).strip() if match else "Electrical Engineering"
        return f"""
        SELECT c.course_name
        FROM courses c
        JOIN departments d ON c.dept_id = d.dept_id
        WHERE lower(d.dept_name) = '{dept.lower()}'
        """

    # 4. How many students in each department?
    if 'how many students' in q and 'each department' in q:
        return """
        SELECT d.dept_name, COUNT(s.student_id) as student_count
        FROM students s
        JOIN departments d ON s.dept_id = d.dept_id
        GROUP BY d.dept_name
        """

    # 5. Who teaches the X course?
    if 'teaches' in q and 'course' in q:
        match = search(r'teaches the ([\w\s]+) course', q)
        cname = match.group(1).strip() if match else "Machine Learning"
        return f"""
        SELECT i.name
        FROM instructors i
        JOIN courses c ON i.instructor_id = c.instructor_id
        WHERE lower(c.course_name) = '{cname.lower()}'
        """

    # 6. List of students with grade A
    if 'students' in q and 'grade a' in q:
        return """
        SELECT s.name
        FROM students s
        JOIN grades g ON s.student_id = g.student_id
        WHERE g.grade = 'A'
        """

    # 7. Average credit hours of courses
    if 'average credit hour' in q or ('average' in q and 'courses' in q):
        return "SELECT AVG(credit_hours) AS avg_credits FROM courses"

    # 8. Student enrollment after a date
    if 'students' in q and 'enrolled after' in q:
        match = search(r'enrolled after ([\d\-]+)', q)
        date = match.group(1) if match else '2024-01-01'
        return f"""
        SELECT s.name, c.course_name, e.enrollment_date
        FROM students s
        JOIN enrollments e ON s.student_id = e.student_id
        JOIN courses c ON e.course_id = c.course_id
        WHERE e.enrollment_date > '{date}'
        """

    # 9. Instructors for a department
    if 'instructors' in q and 'department' in q:
        match = search(r'(\w+ engineering|computer science) department', q)
        dept = match.group(1).strip() if match else 'Mechanical Engineering'
        return f"""
        SELECT i.name
        FROM instructors i
        JOIN departments d ON i.dept_id = d.dept_id
        WHERE lower(d.dept_name) = '{dept.lower()}'
        """

    # 10. For each student, # of courses
    if 'for each student' in q and 'number of courses' in q:
        return """
        SELECT s.name, COUNT(e.course_id) as num_courses
        FROM students s
        LEFT JOIN enrollments e ON s.student_id = e.student_id
        GROUP BY s.name
        """

    # ... Add more patterns as needed!

    # Fallback for basic questions
    if "all students" in q or "name of all students" in q:
        return "SELECT name FROM students"
    if "all courses" in q:
        return "SELECT course_name FROM courses"
    if "departments" in q:
        return "SELECT dept_name FROM departments"
    if "all instructors" in q:
        return "SELECT name FROM instructors"
    # (other patterns...)

    return None  # No match: fallback to model


def natural_language_to_sql(question, schema_info, model, tokenizer):
    """Enhanced NL to SQL with reliable fallbacks"""
    print(f"🔍 Question: {question}")

    # Try reliable pattern matching first
    reliable_sql = get_reliable_sql(question, schema_info)
    if reliable_sql:
        print("🎯 Using reliable pattern-matched SQL")
        return reliable_sql.strip()

    # Fall back to model generation for complex queries
    print("🧠 Using model generation")
    try:
        # Create better schema context
        schema_text = "Database tables:\n"
        for table, info in schema_info.items():
            cols = [col[0] for col in info['columns']]
            schema_text += f"{table}: {', '.join(cols)}\n"

        # Better prompt format
        prompt = f"{schema_text}\nQuestion: {question}\nSQL:"

        inputs = tokenizer(prompt, return_tensors="pt", max_length=400, truncation=True)
        device = next(model.parameters()).device
        inputs = inputs.to(device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=150, num_beams=3, do_sample=False)

        sql = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract SQL part
        if "SQL:" in sql:
            sql = sql.split("SQL:")[-1].strip()

        # Find SQL keywords
        for keyword in ["SELECT", "INSERT", "UPDATE", "DELETE"]:
            if keyword in sql.upper():
                start = sql.upper().find(keyword)
                sql = sql[start:]
                break

        return sql.strip()

    except Exception as e:
        print(f"❌ Model failed: {e}")
        return "SELECT 'Error generating SQL' as message"

def sql_results_to_natural_language(question, sql_query, results_df, nl_model, nl_tokenizer):
    """Enhanced natural language response generation"""
    if len(results_df) == 0:
        return "No results found."

    # Smart response based on question type
    q = question.lower()

    if "how many" in q:
        count = results_df.iloc[0, 0]
        entity = "students" if "students" in q else "courses" if "courses" in q else "instructors" if "instructors" in q else "departments" if "departments" in q else "records"
        return f"There are {count} {entity}."

    elif len(results_df.columns) == 1:
        values = results_df.iloc[:, 0].tolist()
        if "name" in q:
            if len(values) == 1:
                return f"The name is: {values[0]}"
            else:
                names_str = ", ".join(str(v) for v in values[:5])
                more = f" and {len(values) - 5} more" if len(values) > 5 else ""
                return f"The names are: {names_str}{more}"
        else:
            values_str = ", ".join(str(v) for v in values[:3])
            more = f" and {len(values) - 3} more" if len(values) > 3 else ""
            return f"Found: {values_str}{more}"

    else:
        return f"Found {len(results_df)} records with {len(results_df.columns)} columns."

# Test the enhanced functions
print("\n🧪 Testing Enhanced Functions")
print("="*40)

test_questions = [
    "How many students are there?",
    "What courses are available?",
    "Which departments exist?",
    "Show me student names",
    "How many instructors are there?"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n🧪 Test {i}: {question}")

    sql = natural_language_to_sql(question, database_schema, sql_model, sql_tokenizer)
    print(f"🔍 SQL: {sql}")

    try:
        df = pd.read_sql(sql, conn)
        answer = sql_results_to_natural_language(question, sql, df, nl_model, nl_tokenizer)
        print(f"✅ {answer}")
        if len(df) <= 3:
            print(f"📊 Data: {df.to_string(index=False)}")
    except Exception as e:
        print(f"❌ Error: {e}")

    print("-" * 30)

print("\n✅ Enhanced Cell 6 ready!")


🔄 Enhanced Natural Language to SQL Converter

🧪 Testing Enhanced Functions

🧪 Test 1: How many students are there?
🔍 Question: How many students are there?
🧠 Using model generation
🔍 SQL: SELECT COUNT(*) FROM students WHERE NOT student_id IN (SELECT student_id FROM student_id WHERE student_id = 'SQ')
❌ Error: Execution failed on sql 'SELECT COUNT(*) FROM students WHERE NOT student_id IN (SELECT student_id FROM student_id WHERE student_id = 'SQ')': no such table: student_id
------------------------------

🧪 Test 2: What courses are available?
🔍 Question: What courses are available?
🧠 Using model generation
🔍 SQL: SELECT course_name, course_name, course_name FROM databases WHERE NOT course_id IN (SELECT course_id FROM courses WHERE course_name = 'SQ')
❌ Error: Execution failed on sql 'SELECT course_name, course_name, course_name FROM databases WHERE NOT course_id IN (SELECT course_id FROM courses WHERE course_name = 'SQ')': no such table: databases
------------------------------

🧪 Test 

## Cell 7: Complete Question Answering Pipeline

In [ ]:
# Cell 7: Complete Question Answering Pipeline
print("🚀 Complete Question Answering Pipeline")
print("="*50)

def ask_question(question, show_details=True):
    """
    Complete pipeline: Natural Language -> SQL -> Execute -> Natural Language Response
    """
    if show_details:
        print(f"❓ Question: {question}")
        print("="*60)

    try:
        # Step 1: Generate SQL
        if show_details:
            print("🔄 Step 1: Converting natural language to SQL...")

        generated_sql = natural_language_to_sql(
            question, database_schema, sql_model, sql_tokenizer
        )

        if show_details:
            print(f"🔍 Generated SQL: {generated_sql}")

        # Step 2: Validate SQL
        if not any(keyword in generated_sql.upper() for keyword in ['SELECT', 'INSERT', 'UPDATE', 'DELETE']):
            return {
                'question': question,
                'sql': generated_sql,
                'results': pd.DataFrame(),
                'answer': "Invalid SQL generated - no SQL keywords found",
                'success': False
            }

        # Step 3: Execute SQL
        if show_details:
            print("🔄 Step 2: Executing SQL query...")

        results_df = pd.read_sql(generated_sql, conn)

        if show_details:
            print(f"📊 Query executed successfully: {len(results_df)} rows returned")

            if len(results_df) > 0:
                print("📋 Results:")
                display_rows = min(10, len(results_df))
                print(results_df.head(display_rows).to_string(index=False))
                if len(results_df) > display_rows:
                    print(f"... and {len(results_df) - display_rows} more rows")

        # Step 4: Generate natural language response
        if show_details:
            print("🔄 Step 3: Converting results to natural language...")

        nl_response = sql_results_to_natural_language(
            question, generated_sql, results_df, nl_model, nl_tokenizer
        )

        if show_details:
            print(f"🗣️ Final Answer: {nl_response}")
            print("✅ Pipeline completed successfully!")

        return {
            'question': question,
            'sql': generated_sql,
            'results': results_df,
            'answer': nl_response,
            'success': True
        }

    except Exception as e:
        error_msg = f"Error processing question: {str(e)}"
        if show_details:
            print(f"❌ {error_msg}")

        return {
            'question': question,
            'sql': generated_sql if 'generated_sql' in locals() else "Failed to generate",
            'results': pd.DataFrame(),
            'answer': error_msg,
            'success': False
        }

# Test the complete pipeline
print("🚀 Testing Complete Pipeline")
print("="*70)

test_questions = [
    "What are the names of all students?",
    "How many courses are available?",
    "Which students are enrolled in Data Science?",
    "Show me all departments and their locations",
    "Who teaches Machine Learning?"
]

results_log = []

for i, question in enumerate(test_questions, 1):
    print(f"\n🧪 Test {i}")
    print("="*70)

    result = ask_question(question, show_details=True)
    results_log.append(result)

    print(f"\n📊 Test {i} Summary:")
    print(f"   ✅ Success: {result['success']}")
    print(f"   📝 SQL Valid: {len(result['sql']) > 0}")
    print(f"   📊 Results: {len(result['results'])} rows")

    print("\n" + "="*70)

# Performance summary
successful_tests = sum(1 for r in results_log if r['success'])
success_rate = (successful_tests / len(results_log)) * 100

print(f"\n📈 PIPELINE PERFORMANCE SUMMARY")
print("="*70)
print(f"🎯 Success Rate: {success_rate:.1f}% ({successful_tests}/{len(results_log)})")
print(f"📊 Average SQL Length: {sum(len(r['sql']) for r in results_log) / len(results_log):.0f} chars")

avg_results = sum(len(r['results']) for r in results_log) / len(results_log)
print(f"📋 Average Results per Query: {avg_results:.1f} rows")


🚀 Complete Question Answering Pipeline
🚀 Testing Complete Pipeline

🧪 Test 1
❓ Question: What are the names of all students?
🔄 Step 1: Converting natural language to SQL...
🔍 Question: What are the names of all students?
🎯 Using reliable pattern-matched SQL
🔍 Generated SQL: SELECT name FROM students
🔄 Step 2: Executing SQL query...
📊 Query executed successfully: 4 rows returned
📋 Results:
       name
   Amit Rao
 Neha Verma
Rahul Kumar
 Sneha Iyer
🔄 Step 3: Converting results to natural language...
🗣️ Final Answer: The names are: Amit Rao, Neha Verma, Rahul Kumar, Sneha Iyer
✅ Pipeline completed successfully!

📊 Test 1 Summary:
   ✅ Success: True
   📝 SQL Valid: True
   📊 Results: 4 rows


🧪 Test 2
❓ Question: How many courses are available?
🔄 Step 1: Converting natural language to SQL...
🔍 Question: How many courses are available?
🧠 Using model generation
🔍 Generated SQL: SELECT COUNT(*) FROM databases WHERE NOT course_id IN (SELECT course_id FROM courses WHERE course_name = 'SQ')
🔄 S

## Cell 8: Interactive Question Answering Interface

In [ ]:
# Cell 8: Interactive Question Answering Interface WITH EXECUTION TIME
import time

print("🎯 Interactive Question Answering Interface")
print("="*55)

def interactive_qa_session():
    """
    Interactive session for asking questions about your database
    """
    print("🎤 Welcome to Interactive Database Q&A!")
    print("\n📋 Available commands:")
    print("• Ask any question about your database")
    print("• 'help' - Show example questions")
    print("• 'schema' - Show database structure")
    print("• 'quit' - Exit session")
    print("="*55)

    session_count = 0
    successful_queries = 0

    while True:
        try:
            question = input("\n❓ Your question: ").strip()

            if question.lower() in ['quit', 'exit', 'q']:
                print(f"\n👋 Session ended!")
                print(f"📊 Total questions: {session_count}")
                print(f"✅ Successful queries: {successful_queries}")
                if session_count > 0:
                    print(f"🎯 Success rate: {(successful_queries/session_count)*100:.1f}%")
                break

            elif question.lower() == 'help':
                print("\n💡 Example questions you can ask:")
                print("• What are the names of all students?")
                print("• How many courses are available?")
                print("• Which students are enrolled in Data Science?")
                print("• Show me all departments")
                print("• Who teaches the Machine Learning course?")
                print("• What is the average number of credit hours?")
                print("• List students with grade A")
                continue

            elif question.lower() == 'schema':
                print_schema_summary(database_schema)
                continue

            elif len(question) < 3:
                print("⚠️ Please ask a more specific question")
                continue

            session_count += 1
            print("\n" + "="*50)
            t_start = time.time()
            result = ask_question(question, show_details=True)
            t_end = time.time()
            elapsed = t_end - t_start

            if result['success']:
                successful_queries += 1

                # Ask if user wants to see raw data
                if len(result['results']) > 0:
                    show_raw = input("\n🔍 Show raw SQL and data? (y/n): ").strip().lower()
                    if show_raw == 'y':
                        print(f"\n🔍 Raw SQL: {result['sql']}")
                        print(f"📊 Complete Results ({len(result['results'])} rows):")
                        print(result['results'].to_string(index=False))

            print(f"⏱️ Execution Time: {elapsed:.2f} seconds")
            print("="*50)

        except KeyboardInterrupt:
            print("\n\n👋 Session interrupted. Goodbye!")
            break
        except Exception as e:
            print(f"❌ Error: {e}")
            continue

def demo_questions():
    """
    Run a demo with predefined questions
    """
    print("🎬 Running Demo Questions")
    print("="*40)

    demo_questions_list = [
        "How many students are there?",
        "What courses are available?",
        "Which departments exist?",
        "Show me student names",
        "How many instructors are there?"
    ]

    for i, question in enumerate(demo_questions_list, 1):
        print(f"\n🎬 Demo {i}: {question}")
        t_start = time.time()
        result = ask_question(question, show_details=False)
        t_end = time.time()
        elapsed = t_end - t_start

        if result['success']:
            print(f"✅ {result['answer']}")
        else:
            print(f"❌ {result['answer']}")

        print(f"⏱️ Execution Time: {elapsed:.2f} seconds")
        print("-" * 30)

# Choose interaction mode
print("🎯 Choose interaction mode:")
print("1. Interactive session (ask your own questions)")
print("2. Run demo questions")
print("3. Skip to next cell")

while True:
    try:
        choice = input("Enter choice (1-3): ").strip()

        if choice == '1':
            interactive_qa_session()
            break
        elif choice == '2':
            demo_questions()
            break
        elif choice == '3':
            print("⏭️ Skipping to next cell")
            break
        else:
            print("Please enter 1, 2, or 3")
    except KeyboardInterrupt:
        print("\n⏭️ Skipping to next cell")
        break


🎯 Interactive Question Answering Interface
🎯 Choose interaction mode:
1. Interactive session (ask your own questions)
2. Run demo questions
3. Skip to next cell
Enter choice (1-3): 3
⏭️ Skipping to next cell
